In [2]:
import os
import pandas as pd
import brightway2 as bw
import bw2data as bd

In [3]:
BW_PROJECT = 'fasc_new' # insert your project name here
bd.projects.set_current(BW_PROJECT)
list(bd.databases)

['biosphere3', 'ecoinvent-3.10-cutoff', 'metallican_lci_ei', 'Nickel']

# Import additional LCIs

## From MetalliCan and Roy et al (2025)

In [ ]:
lci_dir = r"C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\fasc_metal_sustainability\data\LCIs"
lci_metallican = ["metallican_ei.xlsx", "ni_lci_sofia_roy.xlsx"]

for filename in lci_metallican:
    filepath = os.path.join(lci_dir, filename)
    print(f"Importing: {filepath}")

    importer = bw.ExcelImporter(filepath)
    importer.apply_strategies()

    importer.match_database(
        "ecoinvent-3.10-cutoff",
        fields=("name", "reference product", "unit", "location"),
    )
    importer.match_database(
        "biosphere3",
        fields=("name", "unit", "categories"),
    )

    unlinked = list(importer.unlinked)
    print(f"  → Unlinked exchanges: {len(unlinked)}")

    if not unlinked:
        importer.write_database()
        print(f"  → Database written.")
    else:
        print(f"  → Skipping due to unlinked exchanges.")


In [ ]:
unlinked = list(importer.unlinked)

for i, exc in enumerate(unlinked, start=1):
    print(f"\n--- Unlinked #{i} ---")
    print("type      :", exc.get("type"))
    print("database  :", exc.get("database"))
    print("name      :", exc.get("name"))
    print("ref prod  :", exc.get("reference product"))
    print("location  :", exc.get("location"))
    print("unit      :", exc.get("unit"))
    print("amount    :", exc.get("amount"))
    print("categories:", exc.get("categories"))


## From Lai et al (2025)

# Create inventories for CA for graphite and neodymium

In [ ]:
import uuid
import brightway2 as bw


def find_supplier(db, name, ref_product, unit, locations):
    """
    Cherche un fournisseur par ordre de priorité de localisation.
    """
    for loc in locations:
        for act in db:
            if (
                act.get("name") == name
                and act.get("reference product") == ref_product
                and act.get("unit") == unit
                and act.get("location") == loc
            ):
                return act
    return None


def duplicate_activity_with_new_location(
    source_activity,
    target_location="CA",
    fallback_locations=("RoW", "GLO"),
    new_db_name=None,
    name_suffix=None,  # ex: " (CA)"
):
    """
    Duplique une activité Brightway avec une nouvelle localisation
    et reconnecte les échanges technosphere vers CA, sinon RoW, sinon GLO.
    """

    # --- DB cible ---
    source_db_name = source_activity.key[0]  # ('db_name', 'code')
    target_db_name = new_db_name or source_db_name
    target_db = bw.Database(target_db_name)

    # --- Métadonnées source ---
    src = dict(source_activity)  # dict-like

    # --- Création nouvelle activité ---
    new_code = str(uuid.uuid4())
    new_name = src.get("name")
    if name_suffix:
        new_name = f"{new_name}{name_suffix}"

    new_act = target_db.new_activity(code=new_code)
    # champs essentiels
    new_act["name"] = new_name
    new_act["reference product"] = src.get("reference product")
    new_act["unit"] = src.get("unit")
    new_act["location"] = target_location

    # copie d'autres métadonnées utiles (optionnel)
    for k in ("comment", "categories", "classifications"):
        if k in src:
            new_act[k] = src[k]

    new_act.save()

    # --- Copie / reconnexion des échanges ---
    for exc in source_activity.exchanges():
        exc_type = exc["type"]

        if exc_type == "production":
            new_act.new_exchange(
                input=new_act,
                amount=exc["amount"],
                type="production",
            ).save()

        elif exc_type == "biosphere":
            new_act.new_exchange(
                input=exc.input,
                amount=exc["amount"],
                type="biosphere",
            ).save()

        elif exc_type == "technosphere":
            supplier = exc.input

            new_supplier = find_supplier(
                db=target_db,
                name=supplier.get("name"),
                ref_product=supplier.get("reference product"),
                unit=supplier.get("unit"),
                locations=(target_location,) + tuple(fallback_locations),
            )

            if new_supplier is None:
                new_supplier = supplier  # fallback ultime (garde CN ou autre)

            new_act.new_exchange(
                input=new_supplier,
                amount=exc["amount"],
                type="technosphere",
            ).save()

    return new_act


In [ ]:
IMAGE_20_L = bw.Database("IMAGE_SSP2L20 regionalized")
IMAGE_25_L = bw.Database("IMAGE_SSP2L25 regionalized")
IMAGE_30_L = bw.Database("IMAGE_SSP2L30 regionalized")
IMAGE_35_L = bw.Database("IMAGE_SSP2L35 regionalized")
IMAGE_40_L = bw.Database("IMAGE_SSP2L40 regionalized")

In [61]:
# graphite = next(
#     act for act in IMAGE_20_L
#     if act['name'] == "synthetic graphite production, battery grade"
#     and act['location'] == "CN"
# )

nd = next(
    act for act in IMAGE_20_L
    if act['name'] == "rare earth oxides production, from rare earth carbonate concentrate"
    and act['location'] == "CN"
    and act.get("reference product") == "neodymium oxide"
)

# ca_graphite = duplicate_activity_with_new_location(
#     source_activity=graphite,
#     target_location="CA"
# )

ca_nd = duplicate_activity_with_new_location(
    source_activity=nd,
    target_location="CA"
)

In [62]:
# graphite = next(
#     act for act in IMAGE_25_L
#     if act['name'] == "synthetic graphite production, battery grade"
#     and act['location'] == "CN"
# )

nd = next(
    act for act in IMAGE_25_L
    if act['name'] == "rare earth oxides production, from rare earth carbonate concentrate"
    and act['location'] == "CN"
    and act.get("reference product") == "neodymium oxide"
)

# ca_graphite = duplicate_activity_with_new_location(
#     source_activity=graphite,
#     target_location="CA"
# )

ca_nd = duplicate_activity_with_new_location(
    source_activity=nd,
    target_location="CA"
)

In [63]:
# graphite = next(
#     act for act in IMAGE_30_L
#     if act['name'] == "synthetic graphite production, battery grade"
#     and act['location'] == "CN"
# )

nd = next(
    act for act in IMAGE_30_L
    if act['name'] == "rare earth oxides production, from rare earth carbonate concentrate"
    and act['location'] == "CN"
    and act.get("reference product") == "neodymium oxide"
)

# ca_graphite = duplicate_activity_with_new_location(
#     source_activity=graphite,
#     target_location="CA"
# )

ca_nd = duplicate_activity_with_new_location(
    source_activity=nd,
    target_location="CA"
)

In [64]:
# graphite = next(
#     act for act in IMAGE_35_L
#     if act['name'] == "synthetic graphite production, battery grade"
#     and act['location'] == "CN"
# )

nd = next(
    act for act in IMAGE_35_L
    if act['name'] == "rare earth oxides production, from rare earth carbonate concentrate"
    and act['location'] == "CN"
    and act.get("reference product") == "neodymium oxide"
)

# ca_graphite = duplicate_activity_with_new_location(
#     source_activity=graphite,
#     target_location="CA"
# )

ca_nd = duplicate_activity_with_new_location(
    source_activity=nd,
    target_location="CA"
)

In [65]:
# graphite = next(
#     act for act in IMAGE_40_L
#     if act['name'] == "synthetic graphite production, battery grade"
#     and act['location'] == "CN"
# )

nd = next(
    act for act in IMAGE_40_L
    if act['name'] == "rare earth oxides production, from rare earth carbonate concentrate"
    and act['location'] == "CN"
    and act.get("reference product") == "neodymium oxide"
)

# ca_graphite = duplicate_activity_with_new_location(
#     source_activity=graphite,
#     target_location="CA"
# )

ca_nd = duplicate_activity_with_new_location(
    source_activity=nd,
    target_location="CA"
)